## Recommandation Model Training

#### 1.1 Import Data and Required Packages
##### Importing Pandas, Numpy, Matplotlib, Seaborn and Warings Library.

In [16]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.ensemble import RandomForestClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder


### Load Datasets

In [34]:
features_df = pd.read_csv('../../data/preproccedData/Augmented_PreProccedPhysicalActivityParameters.csv')
targets_df = pd.read_csv('../../data/RecommandationDatasets/gym recommendation.csv')

In [35]:
features_df.head( )

,Id,Age,Gender,Height,Weight,EnergyLevels,Physical_Activity,Sitting_Time,Cardiovascular_Health,Muscle_Strength,Flexibility,Balance,Thirsty,Pain_or_Discomfort,Available_Time,BMI,DiabetesRisk,PhysicalActivityRisk
0,1.070382,24,1,172.867929,48.426644,2.184795,1.894502,0.938458,0.010730,0.000000,0.000000,0.014709,1.812565,0.004581,48.376007,16.508953,30.774236,85.573579
1,3.223005,24,1,170.783864,53.252082,3.292354,0.913221,0.942123,0.979213,0.000000,0.933494,1.073578,0.918528,0.984561,116.117325,18.634367,32.703477,42.791112
2,4.323651,28,0,158.385838,46.480006,4.376199,1.878412,0.909096,0.000201,0.919126,0.945207,1.086765,1.836717,0.008689,290.311582,18.908881,28.756909,14.256016
3,5.421089,24,0,168.730281,54.230813,2.197266,2.828547,0.938141,0.975245,0.006039,0.920186,1.078358,4.570623,0.000000,116.120954,19.418614,37.049640,42.795648
4,6.487447,22,0,159.408917,44.536954,3.276441,2.818626,0.010396,0.006284,0.904967,0.000000,1.080866,2.732837,0.000000,174.169601,17.876292,24.567752,14.254669


### Data Preprocessing

In [36]:
# Drop rows with missing critical values
targets_df.shape

(14589, 15)

#### Head of the Dataset

In [37]:
targets_df.head()

,ID,Sex,Age,Height,Weight,Hypertension,Diabetes,BMI,Level,Fitness Goal,Fitness Type,Exercises,Equipment,Diet,Recommendation
0,1,Male,18,1.68,47.5,No,No,16.83,Underweight,Weight Gain,Muscular Fitness,"Squats, deadlifts, bench presses, and overhead...",Dumbbells and barbells,"Vegetables: (Carrots, Sweet Potato, and Lettuc...",Follow a regular exercise schedule. Adhere to ...
1,2,Male,18,1.68,47.5,Yes,No,16.83,Underweight,Weight Gain,Muscular Fitness,"Squats, deadlifts, bench presses, and overhead...","Light athletic shoes, resistance bands, and li...","Vegetables: (Tomatoes, Garlic, leafy greens, b...",Follow a regular exercise schedule. Adhere to ...
2,3,Male,18,1.68,47.5,No,Yes,16.83,Underweight,Weight Gain,Muscular Fitness,"Squats, yoga, deadlifts, bench presses, and ov...","Dumbbells, barbells and Blood glucose monitor","Vegetables: (Garlic, Roma Tomatoes, Capers and...",Follow a regular exercise schedule. Adhere to ...
3,4,Male,18,1.68,47.5,Yes,Yes,16.83,Underweight,Weight Gain,Muscular Fitness,"Squats, yoga, deadlifts, bench presses, and ov...","Light athletic shoes, resistance bands, light ...","Vegetables: (Garlic, Roma Tomatoes, Capers, Gr...",Follow a regular exercise schedule. Adhere to ...
4,5,Male,18,1.68,47.5,No,No,16.83,Underweight,Weight Gain,Muscular Fitness,"Squats, deadlifts, bench presses, and overhead...",Dumbbells and barbells,"Vegetables: (Carrots, Sweet Potato, Lettuce); ...",Follow a regular exercise schedule. Adhere to ...


### Check Missing values

In [38]:
targets_df.isna().sum()

ID                0
Sex               0
Age               0
Height            0
Weight            0
Hypertension      0
Diabetes          0
BMI               0
Level             0
Fitness Goal      0
Fitness Type      0
Exercises         0
Equipment         0
Diet              0
Recommendation    0
dtype: int64

#### Rename Colums

In [39]:
# Clean all column names
targets_df.columns = targets_df.columns.str.strip()                     # Removes leading/trailing spaces and \n
targets_df.columns = targets_df.columns.str.replace('\n', '', regex=True)  # Removes newlines
targets_df.columns = targets_df.columns.str.replace(' ', '_')           # Optional: Replace spaces with underscores

targets_df.rename(columns={
    'Fitness Goal': 'Fitness_Goal',
    'Fitness Type': 'Fitness_Type',
}, inplace=True)
targets_df.head()

,ID,Sex,Age,Height,Weight,Hypertension,Diabetes,BMI,Level,Fitness_Goal,Fitness_Type,Exercises,Equipment,Diet,Recommendation
0,1,Male,18,1.68,47.5,No,No,16.83,Underweight,Weight Gain,Muscular Fitness,"Squats, deadlifts, bench presses, and overhead...",Dumbbells and barbells,"Vegetables: (Carrots, Sweet Potato, and Lettuc...",Follow a regular exercise schedule. Adhere to ...
1,2,Male,18,1.68,47.5,Yes,No,16.83,Underweight,Weight Gain,Muscular Fitness,"Squats, deadlifts, bench presses, and overhead...","Light athletic shoes, resistance bands, and li...","Vegetables: (Tomatoes, Garlic, leafy greens, b...",Follow a regular exercise schedule. Adhere to ...
2,3,Male,18,1.68,47.5,No,Yes,16.83,Underweight,Weight Gain,Muscular Fitness,"Squats, yoga, deadlifts, bench presses, and ov...","Dumbbells, barbells and Blood glucose monitor","Vegetables: (Garlic, Roma Tomatoes, Capers and...",Follow a regular exercise schedule. Adhere to ...
3,4,Male,18,1.68,47.5,Yes,Yes,16.83,Underweight,Weight Gain,Muscular Fitness,"Squats, yoga, deadlifts, bench presses, and ov...","Light athletic shoes, resistance bands, light ...","Vegetables: (Garlic, Roma Tomatoes, Capers, Gr...",Follow a regular exercise schedule. Adhere to ...
4,5,Male,18,1.68,47.5,No,No,16.83,Underweight,Weight Gain,Muscular Fitness,"Squats, deadlifts, bench presses, and overhead...",Dumbbells and barbells,"Vegetables: (Carrots, Sweet Potato, Lettuce); ...",Follow a regular exercise schedule. Adhere to ...


#### Drop irrelevant Colums

In [40]:
targets_df.drop(columns=["ID"], inplace=True)

In [41]:
targets_df.head()

,Sex,Age,Height,Weight,Hypertension,Diabetes,BMI,Level,Fitness_Goal,Fitness_Type,Exercises,Equipment,Diet,Recommendation
0,Male,18,1.68,47.5,No,No,16.83,Underweight,Weight Gain,Muscular Fitness,"Squats, deadlifts, bench presses, and overhead...",Dumbbells and barbells,"Vegetables: (Carrots, Sweet Potato, and Lettuc...",Follow a regular exercise schedule. Adhere to ...
1,Male,18,1.68,47.5,Yes,No,16.83,Underweight,Weight Gain,Muscular Fitness,"Squats, deadlifts, bench presses, and overhead...","Light athletic shoes, resistance bands, and li...","Vegetables: (Tomatoes, Garlic, leafy greens, b...",Follow a regular exercise schedule. Adhere to ...
2,Male,18,1.68,47.5,No,Yes,16.83,Underweight,Weight Gain,Muscular Fitness,"Squats, yoga, deadlifts, bench presses, and ov...","Dumbbells, barbells and Blood glucose monitor","Vegetables: (Garlic, Roma Tomatoes, Capers and...",Follow a regular exercise schedule. Adhere to ...
3,Male,18,1.68,47.5,Yes,Yes,16.83,Underweight,Weight Gain,Muscular Fitness,"Squats, yoga, deadlifts, bench presses, and ov...","Light athletic shoes, resistance bands, light ...","Vegetables: (Garlic, Roma Tomatoes, Capers, Gr...",Follow a regular exercise schedule. Adhere to ...
4,Male,18,1.68,47.5,No,No,16.83,Underweight,Weight Gain,Muscular Fitness,"Squats, deadlifts, bench presses, and overhead...",Dumbbells and barbells,"Vegetables: (Carrots, Sweet Potato, Lettuce); ...",Follow a regular exercise schedule. Adhere to ...


### Check data types


In [42]:
targets_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14589 entries, 0 to 14588
Data columns (total 14 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Sex             14589 non-null  object 
 1   Age             14589 non-null  int64  
 2   Height          14589 non-null  float64
 3   Weight          14589 non-null  float64
 4   Hypertension    14589 non-null  object 
 5   Diabetes        14589 non-null  object 
 6   BMI             14589 non-null  float64
 7   Level           14589 non-null  object 
 8   Fitness_Goal    14589 non-null  object 
 9   Fitness_Type    14589 non-null  object 
 10  Exercises       14589 non-null  object 
 11  Equipment       14589 non-null  object 
 12  Diet            14589 non-null  object 
 13  Recommendation  14589 non-null  object 
dtypes: float64(3), int64(1), object(10)
memory usage: 1.6+ MB


In [43]:
features_df.head()

,Id,Age,Gender,Height,Weight,EnergyLevels,Physical_Activity,Sitting_Time,Cardiovascular_Health,Muscle_Strength,Flexibility,Balance,Thirsty,Pain_or_Discomfort,Available_Time,BMI,DiabetesRisk,PhysicalActivityRisk
0,1.070382,24,1,172.867929,48.426644,2.184795,1.894502,0.938458,0.010730,0.000000,0.000000,0.014709,1.812565,0.004581,48.376007,16.508953,30.774236,85.573579
1,3.223005,24,1,170.783864,53.252082,3.292354,0.913221,0.942123,0.979213,0.000000,0.933494,1.073578,0.918528,0.984561,116.117325,18.634367,32.703477,42.791112
2,4.323651,28,0,158.385838,46.480006,4.376199,1.878412,0.909096,0.000201,0.919126,0.945207,1.086765,1.836717,0.008689,290.311582,18.908881,28.756909,14.256016
3,5.421089,24,0,168.730281,54.230813,2.197266,2.828547,0.938141,0.975245,0.006039,0.920186,1.078358,4.570623,0.000000,116.120954,19.418614,37.049640,42.795648
4,6.487447,22,0,159.408917,44.536954,3.276441,2.818626,0.010396,0.006284,0.904967,0.000000,1.080866,2.732837,0.000000,174.169601,17.876292,24.567752,14.254669


In [46]:
features_df.shape


(2058, 18)

In [47]:
targets_df.shape

(14589, 14)

In [48]:
combined_df = pd.concat([features_df, targets_df], axis=1)

### # Define features and targets

In [30]:
features = [
    'Age', 'Height', 'Weight', 'EnergyLevels', 'Physical_Activity', 'Sitting_Time',
    'Cardiovascular_Health', 'Muscle_Strength', 'Flexibility', 'Balance', 'Thirsty',
    'Pain_or_Discomfort', 'Available_Time', 'BMI', 'DiabetesRisk', 'PhysicalActivityRisk'
]

targets = ['Level', 'Fitness Goal', 'Fitness Type', 'Exercises', 'Equipment', 'Recommendation']


### Label encode features and targets

In [49]:
label_encoders = {}
for col in features + targets:
    if combined_df[col].dtype == object:
        le = LabelEncoder()
        combined_df[col] = le.fit_transform(combined_df[col].astype(str))
        label_encoders[col] = le

AttributeError: 'DataFrame' object has no attribute 'dtype'

#### Prepare training data

In [33]:
X = gym_encoded[features]
Y = gym_encoded[targets]

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

KeyError: "['EnergyLevels', 'Physical_Activity', 'Sitting_Time', 'Cardiovascular_Health', 'Muscle_Strength', 'Flexibility', 'Balance', 'Thirsty', 'Pain_or_Discomfort', 'Available_Time', 'DiabetesRisk', 'PhysicalActivityRisk'] not in index"

### Score exercises based on user profile

In [17]:
def score_exercises(user):
    goals = get_user_goals(user)
    scores = []
    for _, row in gym_df.iterrows():
        score = 0
        for goal_type, goal_bodypart in goals:
            if goal_type.lower() in row['Type'].lower():
                score += 1
            if goal_bodypart.lower() in row['BodyPart'].lower():
                score += 1
        # Match time availability to difficulty
        if user['Available_Time'] < 60 and row['Level'].lower() == 'beginner':
            score += 1
        if user['Available_Time'] >= 60 and row['Level'].lower() == 'intermediate':
            score += 1
        scores.append(score)
    return np.array(scores)

### Maximal Marginal Relevance (MMR)

In [18]:
def apply_mmr(scores, top_n=5, lambda_param=0.7):
    selected = []
    remaining = list(range(len(scores)))
    similarity_matrix = cosine_similarity(exercise_matrix)

    while len(selected) < top_n and remaining:
        if not selected:
            next_idx = np.argmax(scores[remaining])
            selected.append(remaining.pop(next_idx))
        else:
            mmr_scores = []
            for i in remaining:
                relevance = scores[i]
                diversity = max(similarity_matrix[i][selected]) if selected else 0
                mmr = lambda_param * relevance - (1 - lambda_param) * diversity
                mmr_scores.append(mmr)
            best_idx = remaining[np.argmax(mmr_scores)]
            selected.append(best_idx)
            remaining.remove(best_idx)

    return selected

#### Assign reps/duration for each exercise based on weakness

In [19]:
def assign_workload(user, selected_indices):
    total_time = user['Available_Time']
    plan = []
    
    # Weakness scores: higher = weaker
    muscle_w = 1 - user['Muscle_Strength']
    flex_w = 1 - user['Flexibility']
    balance_w = 1 - user['Balance']
    energy_w = 2.5 - user['EnergyLevels']
    if energy_w < 0: energy_w = 0

    weights = {'Strength': muscle_w, 'Stretching': flex_w, 'Balance': balance_w, 'Cardio': energy_w}
    total_weight = sum(weights.values()) if sum(weights.values()) > 0 else 1

    time_allocs = []
    for idx in selected_indices:
        ex_type = gym_df.iloc[idx]['Type']
        weight = weights.get(ex_type, 0.5)
        portion = weight / total_weight
        time_allocs.append(portion * total_time)

    # Build plan
    for idx, minutes in zip(selected_indices, time_allocs):
        row = gym_df.iloc[idx]
        workload = {
            'Exercise_Title': row['Exercise_Title'],
            'Type': row['Type'],
            'BodyPart': row['BodyPart'],
            'Level': row['Level'],
            'Equipment': row['Equipment'],
            'Recommended_Duration_Minutes': round(minutes, 1),
            'Recommended_Reps': int((minutes * 60) // 30)  # e.g., 1 rep = ~30s
        }
        plan.append(workload)
    
    return pd.DataFrame(plan)

### Recommend exercise plan for a user

In [20]:
def recommend_plan(user_row):
    scores = score_exercises(user_row)
    selected_indices = apply_mmr(scores, top_n=5, lambda_param=0.7)
    plan = assign_workload(user_row, selected_indices)
    return plan

### Example for user 0

In [21]:
user = physical_df.iloc[0]
recommendation_plan = recommend_plan(user)

### Show plan

In [22]:
print(recommendation_plan)


                   Exercise_Title        Type    BodyPart     Level  \
0          Bench barbell roll-out    Strength  Abdominals  Beginner   
1                Dancer's Stretch  Stretching  Lower Back  Beginner   
2                  Stomach Vacuum  Stretching  Abdominals  Beginner   
3  Stiff Leg Barbell Good Morning    Strength  Lower Back  Beginner   
4               Barbell Side Bend    Strength  Abdominals  Beginner   

    Equipment  Recommended_Duration_Minutes  Recommended_Reps  
0     Barbell                          14.7                29  
1  Bodyweight                          14.7                29  
2   Body Only                          14.7                29  
3     Barbell                          14.7                29  
4     Barbell                          14.7                29  
